# Timbre Feature Extraction

Extract per-frame timbre features (and optionally pitch-aligned F0) from every
audio file in a directory.  Concatenate all frames into a single feature matrix
and save it for downstream use (e.g. as a reference distribution for evaluation).

**Configure the cells below, then Run All.**

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().parent.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from evaluation.timbre_metrics import TimbreMetrics
from evaluation.pitch_metrics import PitchMetrics
from utils import load_audio

print(f"Project root: {PROJECT_ROOT}")

## Configuration

Edit **only this cell** to point at a different audio folder, change the
output name, select feature types, or toggle F0 extraction.

In [ ]:
# ── Audio source ──────────────────────────────────────────────────────
AUDIO_DIR = PROJECT_ROOT / "data" / "raw" / "bach-violin" / "audio"
AUDIO_EXTENSIONS = {".wav", ".mp3", ".opus", ".flac", ".ogg"}

# ── Optional metadata CSV ─────────────────────────────────────────────
# Set to None if there is no metadata file.
# If provided, must have a column whose values match the audio filenames.
METADATA_CSV = PROJECT_ROOT / "data" / "raw" / "bach-violin" / "audio.csv"
METADATA_KEY = "filename"          # column in CSV that matches audio filenames
METADATA_COLS = ["collection", "violinist", "work"]  # columns to keep

# ── Output ─────────────────────────────────────────────────────────────
OUT_DIR  = PROJECT_ROOT / "data" / "processed"
OUT_NAME = "bach_violin_timbre_features"   # .parquet and .npz will be appended

# ── Feature extraction parameters ────────────────────────────────────
SR            = 16_000
FRAME_WIDTH   = 0.25        # seconds per analysis frame
OVERLAP       = 0.0         # fraction overlap between frames
FEATURE_TYPES = ["spectral"]   # any of: "spectral", "level", "harmonic"
METRICS       = None        # None = all metrics for selected types;
                            # or a list like ["spectral_centroid", "spectral_spread"]

# ── Pitch (F0) extraction ─────────────────────────────────────────────
EXTRACT_F0            = True
F0_CONFIDENCE_THRESH  = 0.85   # mask frames below this confidence

# ── Sanity-check ──────────────────────────────────────────────────────
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Audio dir : {AUDIO_DIR}  (exists: {AUDIO_DIR.exists()})")
print(f"Metadata  : {METADATA_CSV}  (exists: {METADATA_CSV.exists() if METADATA_CSV else 'N/A'})")
print(f"Output dir: {OUT_DIR}")
print(f"Features  : {FEATURE_TYPES}  |  Metrics filter: {METRICS}")
print(f"Extract F0: {EXTRACT_F0}  |  Confidence threshold: {F0_CONFIDENCE_THRESH}")

## 1. Discover audio files and load metadata

In [ ]:
audio_files = sorted(
    p for p in AUDIO_DIR.rglob("*")
    if p.suffix.lower() in AUDIO_EXTENSIONS
)
print(f"Found {len(audio_files)} audio files")

# Optional metadata
meta_lookup: dict = {}
if METADATA_CSV and Path(METADATA_CSV).exists():
    meta_df = pd.read_csv(METADATA_CSV)
    meta_lookup = meta_df.set_index(METADATA_KEY).to_dict(orient="index")
    print(f"Metadata rows: {len(meta_df)}  |  Columns kept: {METADATA_COLS}")
    display(meta_df.head())
else:
    print("No metadata CSV — only filename will be stored.")

## 2. Extract per-frame features for every file

In [ ]:
tm = TimbreMetrics(sample_rate=SR, frame_width_sec=FRAME_WIDTH, overlap_pct=OVERLAP)
pm = PitchMetrics(sample_rate=SR, frame_width_sec=FRAME_WIDTH, overlap_pct=OVERLAP) if EXTRACT_F0 else None

all_records: list[dict] = []
failed: list[str] = []

for path in tqdm(audio_files, desc="Extracting"):
    try:
        audio, _ = load_audio(path, sr=SR, mono=True)

        # Spectral / level / harmonic features
        series = tm.extract_series_from_array(
            audio, metrics=METRICS, feature_types=FEATURE_TYPES,
        )
        if not series:
            raise ValueError("empty features")

        n_frames = len(next(iter(series.values())))

        # Pitch extraction (aligned to same frame grid)
        if pm is not None:
            f0, conf = pm.extract_f0(audio)
            n = min(n_frames, len(f0))
            series = {k: v[:n] for k, v in series.items()}
            f0 = f0[:n]
            conf = conf[:n]
            f0[conf < F0_CONFIDENCE_THRESH] = np.nan
            series["f0_hz"] = f0
            series["f0_confidence"] = conf
            n_frames = n

    except Exception as e:
        print(f"  SKIP {path.name}: {e}")
        failed.append(path.name)
        continue

    meta = meta_lookup.get(path.name, {})
    all_records.append({
        "filename": path.name,
        "meta":     {c: meta.get(c, "unknown") for c in METADATA_COLS} if meta_lookup else {},
        "series":   series,
        "n_frames": n_frames,
    })

print(f"\nSuccessful: {len(all_records)} / {len(audio_files)}  |  Failed: {len(failed)}")
if failed:
    print("Failed files:", failed)

## 3. Concatenate into a flat DataFrame

In [ ]:
# Feature keys common to every file
common_keys = sorted(
    set.intersection(*[set(r["series"].keys()) for r in all_records])
)
print(f"Features ({len(common_keys)}): {common_keys}")

rows = []
for r in all_records:
    n = r["n_frames"]

    # Metadata columns (always includes filename)
    meta_block = {"filename": [r["filename"]] * n}
    for col, val in r["meta"].items():
        meta_block[col] = [val] * n

    block = pd.DataFrame(meta_block)
    block = pd.concat(
        [block, pd.DataFrame({k: r["series"][k] for k in common_keys})],
        axis=1,
    )
    rows.append(block)

df = pd.concat(rows, ignore_index=True)
print(f"\nDataset shape: {df.shape}  ({df.shape[0]:,} frames \u00d7 {df.shape[1]} columns)")
df.head()

## 4. Save outputs

In [ ]:
parquet_path = OUT_DIR / f"{OUT_NAME}.parquet"
npz_path     = OUT_DIR / f"{OUT_NAME}.npz"

# Feature-only keys (exclude metadata columns)
meta_columns = ["filename"] + (METADATA_COLS if meta_lookup else [])
feat_columns = [c for c in common_keys if c not in meta_columns]

# Parquet — metadata + features
df.to_parquet(parquet_path, index=False)
print(f"Saved parquet: {parquet_path}  ({parquet_path.stat().st_size / 1e6:.1f} MB)")

# NPZ — feature matrix + feature names + per-frame metadata
npz_data = {
    "features":      df[feat_columns].to_numpy(dtype=np.float64),
    "feature_names":  np.array(feat_columns),
    "filename":       df["filename"].to_numpy(),
}
for col in METADATA_COLS if meta_lookup else []:
    if col in df.columns:
        npz_data[col] = df[col].to_numpy()

np.savez_compressed(npz_path, **npz_data)
print(f"Saved npz:     {npz_path}  ({npz_path.stat().st_size / 1e6:.1f} MB)")
print(f"Feature matrix shape: {npz_data['features'].shape}")

## 5. Quick sanity check

In [ ]:
import matplotlib.pyplot as plt

# Summary stats
print("Feature summary (all frames):")
display(df[feat_columns].describe().T.round(4))

# Frames per file (top 20)
print(f"\nFrames per file (showing first 20 / {df['filename'].nunique()}):")
display(df.groupby("filename").size().rename("n_frames").to_frame().head(20))

# Distribution plot for first non-F0 feature
plot_feat = next((c for c in feat_columns if not c.startswith("f0_")), feat_columns[0])

# Pick grouping column: first metadata col if available, else filename
group_col = METADATA_COLS[0] if meta_lookup and METADATA_COLS[0] in df.columns else "filename"
n_groups = df[group_col].nunique()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# KDE per group (cap at 10 groups for readability)
ax = axes[0]
for i, (name, grp) in enumerate(df.groupby(group_col)):
    if i >= 10:
        ax.set_xlabel(f"(showing 10 / {n_groups} groups)")
        break
    grp[plot_feat].plot.kde(ax=ax, label=str(name)[:30])
ax.set_title(f"'{plot_feat}' per {group_col}")
ax.legend(fontsize=7, ncol=2)

# F0 histogram (if extracted)
ax = axes[1]
if "f0_hz" in df.columns:
    f0_valid = df["f0_hz"].dropna()
    ax.hist(f0_valid, bins=80, color="steelblue", edgecolor="white", linewidth=0.3)
    ax.set_xlabel("F0 (Hz)")
    ax.set_title(f"F0 distribution ({len(f0_valid):,} voiced frames)")
else:
    ax.text(0.5, 0.5, "F0 extraction disabled", ha="center", va="center",
            transform=ax.transAxes, fontsize=12, color="grey")
    ax.set_title("F0")

plt.tight_layout()
plt.show()